# Agentic RAG: Retrieve, Grade, Rewrite Once

| Field | Value |
|---|---|
| Stage | LangGraph and agentic RAG |
| Difficulty | Advanced |
| Status | Complete |
| Requires network/API | No |
| Last reviewed | 2026-09-25 |

Callout - Key idea:
Agentic RAG earns its loop only when a measurable retrieval failure triggers a bounded corrective action.

## 30-Second Summary

This notebook builds an offline LangGraph that retrieves, grades lexical evidence, rewrites one vocabulary-mismatched query, and then answers with a citation. The retry budget is one; unsupported queries terminate by abstaining.

## Why This Matters

A fixed RAG chain cannot react when retrieval has no support. An unbounded agent can loop, spend, or hallucinate. Explicit grades, retry budgets, and terminal reasons make correction testable.

## Scope

| Covers | Does not cover |
|---|---|
| Retrieval, support grade, one rewrite, citation, abstention, trace | Hosted LLM, web search, arbitrary planning, hidden reasoning |


## Mental Model

```text
query -> retrieve -> supported? yes -> answer
                    no  -> rewrite (max 1) -> retrieve -> answer/abstain
```


In [1]:
from typing import TypedDict
import re
from langgraph.graph import END, START, StateGraph

DOCUMENTS = [
    {"id": "access", "text": "Role access requires manager approval."},
    {"id": "leave", "text": "Employees receive twenty days of paid leave each year."},
]

def terms(text: str) -> set[str]:
    return set(re.findall(r"[a-z0-9]+", text.lower())) - {"a", "an", "do", "how", "is", "of", "the", "to"}

class RagState(TypedDict, total=False):
    question: str
    query: str
    document_id: str
    document_text: str
    score: int
    retries: int
    answer: str
    terminal_reason: str


## How It Works

Retrieval exposes a support score. The grader routes zero-score results to one deterministic rewrite; a second zero-score result abstains. The answer node can only use the selected document and includes its ID.


## Baseline

The raw query uses `holiday` while the source uses `paid leave`. A one-shot lexical retriever ties at zero and selects the access document by ID.


In [2]:
question = "How much annual holiday do staff get?"

def retrieve_text(query: str) -> tuple[dict, int]:
    ranked = sorted(
        ((document, len(terms(query) & terms(document["text"]))) for document in DOCUMENTS),
        key=lambda item: (-item[1], item[0]["id"]),
    )
    return ranked[0]

baseline_document, baseline_score = retrieve_text(question)
baseline_document, baseline_score


({'id': 'access', 'text': 'Role access requires manager approval.'}, 0)

## Technique Implementation

Each node returns a small state delta. The rewrite uses a governed alias, not an unconstrained answer. Routing depends on score and retry count, not prose reasoning.


In [3]:
def retrieve_node(state: RagState) -> RagState:
    document, score = retrieve_text(state.get("query", state["question"]))
    return {"document_id": document["id"], "document_text": document["text"], "score": score}

def route_after_retrieval(state: RagState) -> str:
    if state["score"] > 0: return "answer"
    if state.get("retries", 0) < 1: return "rewrite"
    return "abstain"

def rewrite_node(state: RagState) -> RagState:
    rewritten = state["question"].lower().replace("annual holiday", "paid leave").replace("staff", "employees")
    return {"query": rewritten, "retries": state.get("retries", 0) + 1}

def answer_node(state: RagState) -> RagState:
    return {"answer": f"{state['document_text']} [{state['document_id']}]", "terminal_reason": "supported"}

def abstain_node(state: RagState) -> RagState:
    return {"answer": "I do not know based on the available documents.", "terminal_reason": "unsupported"}


## Controlled Experiment

We compile the graph and compare the holiday question with an unsupported weather question. The trace must show at most one rewrite and a clear terminal reason.


In [4]:
builder = StateGraph(RagState)
for name, node in (("retrieve", retrieve_node), ("rewrite", rewrite_node), ("answer", answer_node), ("abstain", abstain_node)):
    builder.add_node(name, node)
builder.add_edge(START, "retrieve")
builder.add_conditional_edges("retrieve", route_after_retrieval, {"answer": "answer", "rewrite": "rewrite", "abstain": "abstain"})
builder.add_edge("rewrite", "retrieve")
builder.add_edge("answer", END)
builder.add_edge("abstain", END)
graph = builder.compile()

supported = graph.invoke({"question": question, "query": question, "retries": 0})
unsupported = graph.invoke({"question": "Will it rain tomorrow?", "query": "Will it rain tomorrow?", "retries": 0})
trace = list(graph.stream({"question": question, "query": question, "retries": 0}, stream_mode="updates"))
{"supported": supported, "unsupported": unsupported, "nodes": [next(iter(event)) for event in trace]}


{'supported': {'question': 'How much annual holiday do staff get?',
  'query': 'how much paid leave do employees get?',
  'document_id': 'leave',
  'document_text': 'Employees receive twenty days of paid leave each year.',
  'score': 3,
  'retries': 1,
  'answer': 'Employees receive twenty days of paid leave each year. [leave]',
  'terminal_reason': 'supported'},
 'unsupported': {'question': 'Will it rain tomorrow?',
  'query': 'will it rain tomorrow?',
  'document_id': 'access',
  'document_text': 'Role access requires manager approval.',
  'score': 0,
  'retries': 1,
  'answer': 'I do not know based on the available documents.',
  'terminal_reason': 'unsupported'},
 'nodes': ['retrieve', 'rewrite', 'retrieve', 'answer']}

## Evaluation

The holiday query follows `retrieve → rewrite → retrieve → answer`, retrieves `leave`, and cites it. The weather query uses its single rewrite budget and then abstains. Both terminate deterministically.


In [5]:
assert baseline_document["id"] == "access" and baseline_score == 0
assert supported["document_id"] == "leave" and supported["retries"] == 1 and "[leave]" in supported["answer"]
assert unsupported["terminal_reason"] == "unsupported" and unsupported["retries"] == 1
assert [next(iter(event)) for event in trace] == ["retrieve", "rewrite", "retrieve", "answer"]
print("Agentic RAG correction checks passed.")


Agentic RAG correction checks passed.


## Decision Guide

| Failure | Response |
|---|---|
| Vocabulary mismatch | Bounded rewrite/expansion |
| Corpus lacks answer | Abstain |
| Known fixed sequence | Deterministic graph |
| Several authorized strategies | Bounded router/agent with evaluation |


## Failure Modes and Debugging

| Symptom | Cause | Fix |
|---|---|---|
| Infinite rewrite loop | No retry budget | Hard step/retry limit |
| Wrong doc graded supported | Weak score/threshold | Labeled grader evaluation |
| Unsupported answer | Answer ignores evidence | Evidence-only prompt/check |
| Cost grows silently | Loop not traced | Per-node latency/token/tool budget |


## Production Notes

### Observability
Trace query versions, ranked IDs/scores, grade, retry count, node sequence, citation, and terminal reason.

### Safety and Guardrails
Rewrites inherit authorization scope and cannot introduce unrestricted sources.

### Latency and Cost
Budget the worst-case loop and keep a deterministic abstention path.


## Practice

Add a misspelling rewrite and prove the graph never executes more than one corrective pass.

## Recall

Toggle - Recall: When is a loop justified?
When a measurable failure can trigger a bounded action that improves outcomes.

Toggle - Recall: What ends this graph?
Supported evidence or an exhausted retry budget followed by abstention.

## Sources

- [LangGraph workflows and agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents)
- Repository-owned synthetic policy fixture

## Review Log

| Date | Status | Confidence | Next review focus |
|---|---|---|---|
| 2026-09-25 | Complete; executed and visually reviewed | High for the bounded offline workflow | Evaluate learned graders and rewrite quality |
